In [42]:
!pip -q install chromadb openai langchain tiktoken


[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: C:\Users\ADMIN\AppData\Local\Programs\Python\Python311\python.exe -m pip install --upgrade pip


In [43]:
!pip show chromadb

Name: chromadb
Version: 1.5.8
Summary: Chroma.
Home-page: https://github.com/chroma-core/chroma
Author: 
Author-email: Jeff Huber <jeff@trychroma.com>, Anton Troynikov <anton@trychroma.com>
License: 
Location: C:\Users\ADMIN\AppData\Local\Programs\Python\Python311\Lib\site-packages
Requires: bcrypt, build, grpcio, httpx, importlib-resources, jsonschema, kubernetes, mmh3, numpy, onnxruntime, opentelemetry-api, opentelemetry-exporter-otlp-proto-grpc, opentelemetry-sdk, orjson, overrides, pybase64, pydantic, pydantic-settings, pypika, pyyaml, rich, tenacity, tokenizers, tqdm, typer, typing-extensions, uvicorn
Required-by: 


In [44]:
import requests

url = "https://www.dropbox.com/s/vs6ocyvpzzncvwh/new_articles.zip?dl=1"

response = requests.get(url)

with open("new_articles.zip", "wb") as file:
    file.write(response.content)

print("Download complete")

Download complete


In [45]:
import zipfile

with zipfile.ZipFile("new_articles.zip", "r") as zip_ref:
    zip_ref.extractall("new_articles")

print("Files extracted successfully")

Files extracted successfully


In [46]:
pip install python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [47]:
from dotenv import load_dotenv
import os

load_dotenv()

openai_key = os.getenv("OPENAI_API_KEY")

In [48]:
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_openai import OpenAI
from langchain_community.document_loaders import DirectoryLoader
from langchain_community.document_loaders import TextLoader
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_classic.chains import RetrievalQA

In [49]:
loader = DirectoryLoader(
    r"D:\Vector_DataBase\new_articles",
    glob="./*.txt",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"}
)

documents = loader.load()

In [50]:
loader = DirectoryLoader(
    r"D:\Vector_DataBase\new_articles",
    glob="./*.txt",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "latin-1"}
)

documents = loader.load()

In [51]:
loader.load()

[Document(metadata={'source': 'D:\\Vector_DataBase\\new_articles\\05-03-ai-powered-supply-chain-startup-pando-lands-30m-investment.txt'}, page_content='Signaling that investments in the supply chain sector remain robust, Pando, a startup developing fulfillment management technologies, today announced that it raised $30 million in a Series B round, bringing its total raised to $45 million.\n\nIron Pillar and Uncorrelated Ventures led the round, with participation from existing investors Nexus Venture Partners, Chiratae Ventures and Next47. CEO and founder Nitin Jayakrishnan says that the new capital will be put toward expanding Pandoâ\x80\x99s global sales, marketing and delivery capabilities.\n\nâ\x80\x9cWe will not expand into new industries or adjacent product areas,â\x80\x9d he told TechCrunch in an email interview. â\x80\x9cGreat talent is the foundation of the business â\x80\x94 we will continue to augment our teams at all levels of the organization. Pando is also open to explorin

In [52]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

text = text_splitter.split_documents(documents)

print(text[:2])

[Document(metadata={'source': 'D:\\Vector_DataBase\\new_articles\\05-03-ai-powered-supply-chain-startup-pando-lands-30m-investment.txt'}, page_content='Signaling that investments in the supply chain sector remain robust, Pando, a startup developing fulfillment management technologies, today announced that it raised $30 million in a Series B round, bringing its total raised to $45 million.\n\nIron Pillar and Uncorrelated Ventures led the round, with participation from existing investors Nexus Venture Partners, Chiratae Ventures and Next47. CEO and founder Nitin Jayakrishnan says that the new capital will be put toward expanding Pandoâ\x80\x99s global sales, marketing and delivery capabilities.\n\nâ\x80\x9cWe will not expand into new industries or adjacent product areas,â\x80\x9d he told TechCrunch in an email interview. â\x80\x9cGreat talent is the foundation of the business â\x80\x94 we will continue to augment our teams at all levels of the organization. Pando is also open to explorin

In [53]:
len(text)

236

# Create Database

In [54]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma

In [55]:
persist_directory = "db"

In [58]:
%pip install chromadb

  Using cached chromadb-1.5.8-cp39-abi3-win_amd64.whl.metadata (5.1 kB)
  Using cached build-1.4.4-py3-none-any.whl.metadata (5.8 kB)
  Using cached opentelemetry_api-1.41.1-py3-none-any.whl.metadata (1.5 kB)
  Using cached opentelemetry_exporter_otlp_proto_grpc-1.41.1-py3-none-any.whl.metadata (2.6 kB)
  Using cached opentelemetry_sdk-1.41.1-py3-none-any.whl.metadata (1.7 kB)
  Using cached pypika-0.51.1-py2.py3-none-any.whl.metadata (51 kB)
  Using cached overrides-7.7.0-py3-none-any.whl.metadata (5.8 kB)
  Using cached importlib_resources-7.1.0-py3-none-any.whl.metadata (4.0 kB)
  Using cached bcrypt-5.0.0-cp39-abi3-win_amd64.whl.metadata (10 kB)
  Using cached typer-0.25.0-py3-none-any.whl.metadata (15 kB)
  Using cached kubernetes-35.0.0-py2.py3-none-any.whl.metadata (1.7 kB)
  Using cached pyproject_hooks-1.2.0-py3-none-any.whl.metadata (1.3 kB)
  Using cached websocket_client-1.9.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached requests_oauthlib-2.0.0-py2.py3-none-any.whl.met

In [59]:
embedding = OpenAIEmbeddings()

vectordb = Chroma.from_documents(
    documents=text,
    embedding=embedding,
    persist_directory="db"
)